In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install sentence-transformers torch matplotlib scipy scikit-learn -q

# Experiment Overview

Experiment 13 — Blind Decoding Audit

Motivation:
  Exp 8 shows that compressed vectors at k=16 decode cleanly to correct
  words in multiple languages. But the CLAIM that the dense channel is
  "human-auditable" needs a harder test:
  
  Generate compressed vectors for held-out concepts. Decode each to
  top-5 nearest words across 4 languages. Give only those word lists
  (no concept label) and ask: can you identify the concept?
  
  If hit rate is high: the channel IS genuinely interpretable.
  If not: the nearest-word decoding is misleading and the interpretability
  claim from Exp 8 needs to be revised.

  This converts an engineering claim into an empirical one and is
  directly relevant to the AI alignment angle about maintaining human
  oversight of AI-AI communication.

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('LaBSE')
print(f'LaBSE loaded, device={DEVICE}')

## SECTION 1: Concept vocabulary (train + held-out split)

In [ ]:
TRAIN_CONCEPTS = [
    'water', 'fire',  'earth',    'sky',      'love',
    'fear',  'trust', 'light',    'dark',     'time',
    'life',  'death', 'eat',      'sleep',    'run',
    'give',  'take',  'speak',    'think',    'feel',
    'wind',  'stone', 'river',    'mountain', 'forest',
    'child', 'elder', 'friend',   'enemy',    'peace',
    'war',   'pain',  'joy',      'hope',     'dream',
    'build', 'break', 'find',     'lose',     'change',
]

# 20 held-out concepts — NEVER seen during bottleneck training
HELD_OUT_CONCEPTS = [
    'sun',     'moon',    'rain',    'snow',    'ocean',
    'blood',   'bread',   'bird',    'tree',    'door',
    'king',    'mother',  'song',    'dance',   'truth',
    'anger',   'wisdom',  'shadow',  'journey', 'silence',
]

ALL_CONCEPTS = TRAIN_CONCEPTS + HELD_OUT_CONCEPTS

# Multilingual interpretability vocabulary (large set for decoding)
INTERP_VOCAB = {
    'en': ALL_CONCEPTS + [
        'happiness','sadness','cold','warm','bright','day','night','road',
        'sword','shield','garden','sea','island','cloud','star','power',
        'strength','knowledge','family','woman','man','wolf','lion','horse',
        'music','color','gold','silver','iron','salt','sugar','city','village',
        'law','justice','mercy','god','devil','heaven','hell','ghost',
        'hunger','thirst','work','play','laugh','cry','hate','love',
    ],
    'es': [
        'agua','fuego','tierra','cielo','amor','miedo','confianza','luz',
        'oscuridad','tiempo','vida','muerte','comer','dormir','correr','dar',
        'tomar','hablar','pensar','sentir','viento','piedra','río','montaña',
        'bosque','niño','anciano','amigo','enemigo','paz','guerra','dolor',
        'alegría','esperanza','sueño','construir','romper','encontrar','perder','cambiar',
        'sol','luna','lluvia','nieve','océano','sangre','pan','pájaro',
        'árbol','puerta','rey','madre','canción','baile','verdad','ira',
        'sabiduría','sombra','viaje','silencio','felicidad','tristeza','frío',
        'calor','día','noche','camino','espada','jardín','mar','isla',
        'nube','estrella','poder','familia','mujer','hombre','lobo','león',
    ],
    'zh': [
        '水','火','土','天空','爱','恐惧','信任','光','暗','时间',
        '生命','死亡','吃','睡觉','跑','给','拿','说话','思考','感觉',
        '风','石头','河流','山','森林','孩子','老人','朋友','敌人','和平',
        '战争','痛苦','喜悦','希望','梦想','建造','破坏','找到','失去','改变',
        '太阳','月亮','雨','雪','海洋','血','面包','鸟','树','门',
        '国王','母亲','歌曲','舞蹈','真理','愤怒','智慧','影子','旅程','沉默',
        '幸福','悲伤','冷','温暖','明亮','白天','夜晚','道路','剑','花园',
        '海','岛','云','星星','力量','家庭','女人','男人','狼','狮子',
    ],
    'ar': [
        'ماء','نار','أرض','سماء','حب','خوف','ثقة','ضوء','ظلام','وقت',
        'حياة','موت','أكل','نوم','ركض','أعطى','أخذ','تكلم','فكر','شعر',
        'ريح','حجر','نهر','جبل','غابة','طفل','مسن','صديق','عدو','سلام',
        'حرب','ألم','فرح','أمل','حلم','بنى','كسر','وجد','فقد','تغير',
        'شمس','قمر','مطر','ثلج','محيط','دم','خبز','طائر','شجرة','باب',
        'ملك','أم','أغنية','رقص','حقيقة','غضب','حكمة','ظل','رحلة','صمت',
    ],
}

print(f'Train concepts: {len(TRAIN_CONCEPTS)}')
print(f'Held-out concepts: {len(HELD_OUT_CONCEPTS)}')
print(f'Interp vocab sizes: {", ".join(f"{k}:{len(v)}" for k,v in INTERP_VOCAB.items())}')

## SECTION 2: Train bottleneck encoder (same as Exp 8 v2)

In [ ]:
class BottleneckEncoder(nn.Module):
    def __init__(self, input_dim=768, bottleneck_dim=8, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, bottleneck_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, hidden), nn.GELU(),
            nn.Linear(hidden, input_dim),
        )

    def encode(self, x): return self.encoder(x)
    def decode(self, z):
        return F.normalize(self.decoder(z), dim=-1)
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def train_bottleneck(k, concept_embs, n_epochs=300, batch_size=64, lr=1e-3, device=DEVICE):
    X = torch.tensor(concept_embs, dtype=torch.float32).to(device)
    N = len(X)
    model = BottleneckEncoder(768, k).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(N, device=device)[:batch_size]
        targets = X[idx]
        recon, z = model(targets)
        
        recon_loss = 1.0 - (recon * targets).sum(dim=-1).mean()
        
        distractor_idx = torch.stack([
            torch.randperm(N, device=device)[:10] for _ in range(len(idx))
        ])
        distractor_idx[:, 0] = idx
        candidates = X[distractor_idx]
        query = recon.unsqueeze(1)
        scores = (candidates * query).sum(dim=-1)
        labels = torch.zeros(len(idx), dtype=torch.long, device=device)
        ref_loss = F.cross_entropy(scores, labels)
        
        (recon_loss + ref_loss).backward()
        opt.step(); opt.zero_grad()
    
    return model

# Embed training concepts and train encoder at k=16
print('Embedding training concepts...')
train_embs = embedder.encode(TRAIN_CONCEPTS, normalize_embeddings=True)
all_embs = embedder.encode(ALL_CONCEPTS, normalize_embeddings=True)
holdout_embs = all_embs[len(TRAIN_CONCEPTS):]

K_TEST = 16
print(f'Training bottleneck encoder at k={K_TEST}...')
enc_model = train_bottleneck(K_TEST, train_embs)
print('Training complete.')

# Build interpretability index
print('Building multilingual interpretability index...')
interp_records = []
for lang, words in INTERP_VOCAB.items():
    embs = embedder.encode(words, normalize_embeddings=True)
    for word, emb in zip(words, embs):
        interp_records.append({'lang': lang, 'word': word, 'emb': emb})

interp_embs = np.stack([r['emb'] for r in interp_records])
interp_words = [r['word'] for r in interp_records]
interp_langs = [r['lang'] for r in interp_records]
print(f'Index: {len(interp_words)} entries')

## SECTION 3: Blind Decoding Audit

In [ ]:
print('\n' + '═'*60)
print('BLIND DECODING AUDIT')
print('═'*60)

def decode_to_words(concept_emb, model, top_k=5, device=DEVICE):
    """Compress a concept embedding and decode to nearest words per language."""
    X = torch.tensor(concept_emb, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        recon, z = model(X)
        recon_np = recon.cpu().numpy()[0]
    
    sims = interp_embs @ recon_np
    results = {}
    for lang in set(interp_langs):
        lang_mask = np.array(interp_langs) == lang
        lang_sims = sims.copy()
        lang_sims[~lang_mask] = -999
        top_idx = np.argsort(-lang_sims)[:top_k]
        results[lang] = [(interp_words[i], float(sims[i])) for i in top_idx]
    return results, z.cpu().numpy()[0]


def blind_identify(word_lists, all_concept_labels, embedder):
    """
    Given decoded word lists (no concept label), try to identify the concept.
    Strategy: embed all decoded words, compute centroid, find nearest concept.
    """
    all_words = []
    for lang, words_sims in word_lists.items():
        all_words.extend([w for w, s in words_sims])
    
    if not all_words:
        return None, 0.0
    
    word_embs = embedder.encode(all_words, normalize_embeddings=True)
    centroid = word_embs.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    
    concept_embs = embedder.encode(all_concept_labels, normalize_embeddings=True)
    sims = concept_embs @ centroid
    best_idx = int(sims.argmax())
    return all_concept_labels[best_idx], float(sims[best_idx])


# Run audit on held-out concepts
print(f'\nDecoding {len(HELD_OUT_CONCEPTS)} held-out concepts at k={K_TEST}:')
print(f'(These concepts were NEVER seen during encoder training)\n')

audit_results = []
for i, concept in enumerate(HELD_OUT_CONCEPTS):
    emb = holdout_embs[i]
    word_lists, z_vec = decode_to_words(emb, enc_model, top_k=5)
    
    # Print the word list (what a human auditor would see)
    print(f'  Concept #{i+1} (hidden): decoded words =')
    for lang in sorted(word_lists.keys()):
        top_words = ', '.join([f'{w}({s:.2f})' for w, s in word_lists[lang][:3]])
        print(f'    {lang}: {top_words}')
    
    # Blind identification
    identified, confidence = blind_identify(word_lists, ALL_CONCEPTS, embedder)
    correct = (identified == concept)
    audit_results.append({
        'concept': concept,
        'identified': identified,
        'correct': correct,
        'confidence': confidence,
        'word_lists': {lang: [(w, s) for w, s in wl[:3]] for lang, wl in word_lists.items()},
    })
    
    status = '✓' if correct else f'✗ (guessed: {identified})'
    print(f'    → Blind ID: {status}  (confidence: {confidence:.3f})')
    print()

## SECTION 4: Audit across multiple k values

In [ ]:
print('═'*60)
print('AUDIT ACROSS BOTTLENECK DIMENSIONS')
print('═'*60)

K_VALUES = [2, 4, 8, 16, 32, 64]
k_audit_results = {}

for k in K_VALUES:
    print(f'\n  Training k={k}...', end=' ', flush=True)
    m = train_bottleneck(k, train_embs, n_epochs=300)
    
    correct_count = 0
    top3_count = 0
    
    for i, concept in enumerate(HELD_OUT_CONCEPTS):
        emb = holdout_embs[i]
        word_lists, _ = decode_to_words(emb, m, top_k=5)
        
        # Top-1 blind identification
        identified, conf = blind_identify(word_lists, ALL_CONCEPTS, embedder)
        if identified == concept:
            correct_count += 1
        
        # Top-3: check if concept is in top 3 nearest
        all_words = []
        for lang, wl in word_lists.items():
            all_words.extend([w for w, s in wl])
        word_embs_k = embedder.encode(all_words, normalize_embeddings=True)
        centroid_k = word_embs_k.mean(axis=0)
        centroid_k /= np.linalg.norm(centroid_k)
        concept_embs_k = embedder.encode(ALL_CONCEPTS, normalize_embeddings=True)
        sims_k = concept_embs_k @ centroid_k
        top3_concepts = [ALL_CONCEPTS[j] for j in np.argsort(-sims_k)[:3]]
        if concept in top3_concepts:
            top3_count += 1
    
    hit_rate = correct_count / len(HELD_OUT_CONCEPTS)
    top3_rate = top3_count / len(HELD_OUT_CONCEPTS)
    k_audit_results[k] = {'hit_rate': hit_rate, 'top3_rate': top3_rate}
    print(f'Hit@1 = {hit_rate:.3f}  Hit@3 = {top3_rate:.3f}')

## SECTION 5: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Hit rates across k
ks = list(k_audit_results.keys())
hit1 = [k_audit_results[k]['hit_rate'] for k in ks]
hit3 = [k_audit_results[k]['top3_rate'] for k in ks]
axes[0].plot(ks, hit1, 'o-', color='#1976D2', linewidth=2, markersize=8, label='Hit@1')
axes[0].plot(ks, hit3, 's-', color='#1D9E75', linewidth=2, markersize=8, label='Hit@3')
axes[0].axhline(1/len(ALL_CONCEPTS), color='gray', linestyle='--', alpha=0.5, label='chance')
axes[0].set_xscale('log', base=2)
axes[0].set_xlabel('Bottleneck dimension k')
axes[0].set_ylabel('Blind identification rate')
axes[0].set_title('Blind Decoding Audit:\nCan decoded word lists identify the concept?')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, 1.05)

# Plot 2: Per-concept results at k=16
ax2 = axes[1]
concepts_sorted = sorted(audit_results, key=lambda x: x['confidence'], reverse=True)
colors = ['#1D9E75' if r['correct'] else '#E24B4A' for r in concepts_sorted]
ax2.barh([r['concept'] for r in concepts_sorted],
         [r['confidence'] for r in concepts_sorted],
         color=colors, height=0.6)
ax2.set_xlabel('Identification confidence')
ax2.set_title(f'Per-concept results at k={K_TEST}\n(green=correct, red=wrong)')
ax2.tick_params(labelsize=8)

# Plot 3: Confusion examples
ax3 = axes[2]
wrong = [r for r in audit_results if not r['correct']]
if wrong:
    wrong_text = '\n'.join([
        f'{r["concept"]} → {r["identified"]}' for r in wrong[:10]
    ])
    ax3.text(0.1, 0.5, f'Misidentified concepts:\n\n{wrong_text}',
             transform=ax3.transAxes, fontsize=11, verticalalignment='center',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#FFE0E0', alpha=0.8))
    ax3.set_title(f'Failure analysis at k={K_TEST}')
else:
    ax3.text(0.5, 0.5, 'All concepts correctly identified!',
             transform=ax3.transAxes, fontsize=14, ha='center', va='center',
             bbox=dict(boxstyle='round', facecolor='#E0FFE0', alpha=0.8))
    ax3.set_title(f'No failures at k={K_TEST}')
ax3.axis('off')

plt.suptitle('Experiment 13 — Blind Decoding Audit\nIs the dense communication channel genuinely human-auditable?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp13_blind_decoding_audit.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
hit_16 = k_audit_results.get(K_TEST, {})
print('\n' + '═'*60)
print('EXPERIMENT 13 — SUMMARY')
print('═'*60)
print(f'Held-out concepts tested: {len(HELD_OUT_CONCEPTS)}')
print(f'At k={K_TEST}: Hit@1 = {hit_16.get("hit_rate", "N/A"):.3f}  Hit@3 = {hit_16.get("top3_rate", "N/A"):.3f}')
correct_at_16 = sum(1 for r in audit_results if r['correct'])
print(f'  Correctly identified: {correct_at_16}/{len(HELD_OUT_CONCEPTS)}')
print()
print('Interpretation:')
print('  High hit rate (>0.7): Dense channel IS human-auditable.')
print('    → Compressed vectors can be decoded to meaningful word lists')
print('    → Supports the alignment argument for interpretable AI-AI communication')
print('  Low hit rate (<0.5): Nearest-word decoding is MISLEADING.')
print('    → The compressed vector preserves identity but not in a human-readable way')
print('    → The interpretability claim from Exp 8 needs revision')
print()
print('  Key nuance: hit rate should be tested at the k where the protocol')
print('  is actually useful (k=4-16). If interpretability degrades at useful k,')
print('  the information-efficiency vs interpretability tradeoff is real.')